# 02 — Transfer Learning: Fish Detection (Faster R-CNN)

Train a fish-only underwater detector from the unified FOD detection manifests.

Planned model: TorchVision Faster R-CNN (single class: fish).

Source reference for approach: https://pytorch.org/tutorials/intermediate/torchvision_tutorial.html

In [1]:
from pathlib import Path
import torch

ROOT = Path('..').resolve()
MANIFEST_DIR = (ROOT / 'data' / 'manifests').resolve()
OUT_DIR = (ROOT / 'outputs' / 'detection_frcnn').resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('MANIFEST_DIR:', MANIFEST_DIR)
print('OUT_DIR:', OUT_DIR)

device = torch.device(
    'cuda'
    if torch.cuda.is_available()
    else 'mps'
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()
    else 'cpu'
)
print('device:', device)

MANIFEST_DIR: /Users/sushantshah/Desktop/Fish Codes/data/manifests
OUT_DIR: /Users/sushantshah/Desktop/Fish Codes/outputs/detection_frcnn
device: mps


In [2]:
import sys

sys.path.append(str((ROOT / 'src').resolve()))

from fish_ai.data.detection_dataset import FishDetectionDataset
from fish_ai.train.detection_frcnn import TrainConfig, collate_fn, fit, simple_box_recall
from torch.utils.data import DataLoader

train_manifest = MANIFEST_DIR / 'fod_detection_train.jsonl'
val_manifest = MANIFEST_DIR / 'fod_detection_val.jsonl'

print('train_manifest exists:', train_manifest.exists())
print('val_manifest exists:', val_manifest.exists())

train_manifest exists: True
val_manifest exists: True


## Train

This expects manifests written by `01_data_build_manifests.ipynb`:
- `data/manifests/fod_detection_train.jsonl`
- `data/manifests/fod_detection_val.jsonl`

If you haven’t built manifests yet, run notebook 01 first.

In [ ]:
ds_train = FishDetectionDataset(train_manifest)
ds_val = FishDetectionDataset(val_manifest) if val_manifest.exists() else None

cfg = TrainConfig(num_epochs=5, batch_size=2, num_workers=2, lr=5e-4)
model = fit(ds_train, dataset_val=ds_val, cfg=cfg, device=device)

ckpt_path = OUT_DIR / 'frcnn_fish_only.pt'
torch.save({'model_state': model.state_dict(), 'cfg': cfg.__dict__}, ckpt_path)
print('saved:', ckpt_path)

## Quick eval (sanity metric)

This computes a lightweight recall@IoU=0.5 metric (not COCO mAP yet).

In [ ]:
if ds_val is not None:
    dl_val = DataLoader(ds_val, batch_size=2, shuffle=False, num_workers=2, collate_fn=collate_fn)
    metrics = simple_box_recall(model, dl_val, device=device, iou_thresh=0.5, score_thresh=0.5)
    print(metrics)
else:
    print('No val manifest found; skipping eval')

## COCO box mAP (proper)

This requires a COCO GT JSON file that matches the image ids used in the manifests.
For FOD, point this at the same `instances.json` you used to build manifests.

In [ ]:
from fish_ai.eval.coco_box_eval import evaluate_coco_boxes

# Update this path to match your local FOD annotation location
coco_gt_json = (ROOT / 'data' / 'raw' / 'fod' / 'annotations' / 'instances.json').resolve()
print('coco_gt_json exists:', coco_gt_json.exists())

if ds_val is not None and coco_gt_json.exists():
    dl_val = DataLoader(ds_val, batch_size=2, shuffle=False, num_workers=2, collate_fn=collate_fn)
    ap = evaluate_coco_boxes(model, dl_val, coco_gt_json, device=device, score_thresh=0.05, max_images=500)
    ap
else:
    print('Need val manifest + COCO GT JSON to compute mAP')